# **Зонд и activation patching: практика**

В уроке «Иллюстративные примеры MI» кейс Othello-GPT различал две вещи: представление доски *декодируется* из скрытых состояний (это показал зонд) и модель *пользуется* им (это показало вмешательство). В уроке «Методы механистической интерпретируемости» та же пара стоит на карте методов как чтение и вмешательство. Здесь оба метода применяются к одной задаче и одной модели, и их ответы сравниваются слой за слоем.

Модель — GPT-2 small из TransformerLens (12 слоев, residual stream размерности 768), задача — IOI из кейса 3: в промпте «When Mary and John went to the store, John gave a drink to» правильное продолжение — « Mary» (indirect object, IO), а повторенное имя « John» — субъект (S).

В этой практике вы:

* посчитаете, насколько уверенно модель решает IOI на 200 промптах: разность логитов IO и S;
* обучите линейный зонд на residual stream каждого слоя на двух позициях — на втором упоминании субъекта (S2) и на последнем токене (END) — и найдете, с какого слоя зонд различает, какое имя в промпте IO;
* сделаете activation patching тех же активаций и найдете, на каких слоях и позициях они причинно нужны для ответа;
* сравните две кривые и найдете слои, где информация декодируется, но модель ее отсюда не берет.

Задания пронумерованы, ответы на них вводятся в уроке «Зонд и activation patching: Практика».

## **Подготовка окружения**
Хватит CPU: модель небольшая, и все вычисления здесь — прямые проходы и логистическая регрессия.

In [ ]:
!pip install transformer-lens==3.9.0 scikit-learn -q

In [ ]:
import random
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from transformer_lens import HookedTransformer

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_grad_enabled(False)  # здесь только прямые проходы
model = HookedTransformer.from_pretrained("gpt2-small", device="cpu")
print(model.cfg.n_layers, model.cfg.d_model)

## **Промпты IOI**

Соберем 200 промптов по одному шаблону. Имена, места и предметы — однотокенные слова, поэтому позиции токенов во всех промптах совпадают: S2 — позиция 10, END — последняя позиция 14 (позиция 0 — служебный токен `<|endoftext|>`).

В половине промптов IO — первое из двух имен («When Mary and John …, John gave … to»), в другой половине — второе («When Mary and John …, Mary gave … to»). Метка `y` = 1, если IO — первое имя. Ее и будет предсказывать зонд.

Для activation patching к каждому промпту нужна испорченная пара: тот же промпт, где повторено другое имя. В ней IO и S меняются ролями, и правильный ответ меняется на другое имя.

In [ ]:
NAMES = [" Mary", " John", " Alice", " Bob", " Sarah", " Tom", " Anna", " James", " Emma", " David",
         " Kate", " Paul", " Laura", " Mark", " Lisa", " Peter", " Jane", " Mike", " Susan", " Steve"]
PLACES = [" store", " park", " school", " garden", " station", " office"]
OBJECTS = [" drink", " book", " ring", " letter", " ball", " gift"]
TEMPLATE = "When{A} and{B} went to the{P},{S} gave a{O} to"
N = 200

rows = []
for i in range(N):
    a, b = random.sample(NAMES, 2)
    io_first = i % 2 == 0  # в половине промптов IO — первое имя
    rows.append(dict(A=a, B=b, P=random.choice(PLACES), O=random.choice(OBJECTS),
                     S=b if io_first else a, IO=a if io_first else b, y=int(io_first)))

clean_tokens = model.to_tokens([TEMPLATE.format(**r) for r in rows])
corr_tokens = model.to_tokens([TEMPLATE.format(**{**r, "S": r["IO"]}) for r in rows])  # испорченная пара: повторено другое имя
io_tok = torch.tensor([model.to_single_token(r["IO"]) for r in rows])
s_tok = torch.tensor([model.to_single_token(r["S"]) for r in rows])
y = np.array([r["y"] for r in rows])
S2, END = 10, clean_tokens.shape[1] - 1
print(model.to_str_tokens(clean_tokens[0]))
print(model.to_str_tokens(corr_tokens[0]))
print("S2:", model.to_str_tokens(clean_tokens[0])[S2], " END:", model.to_str_tokens(clean_tokens[0])[END])

## **Насколько уверенно модель решает IOI**

Мера — разность логитов на последней позиции: $\mathrm{LD} = \mathrm{logit}(\mathrm{IO}) - \mathrm{logit}(\mathrm{S})$, где IO и S берутся из *чистого* промпта. На чистых промптах она положительна, если модель выбирает IO; на испорченных — отрицательна, потому что там правильный ответ — S чистого промпта.

Допишите функцию `logit_diff`.

In [ ]:
def logit_diff(logits):
    last = logits[:, -1]
    return ...  # Ваш код здесь


clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corr_logits = model(corr_tokens)
ld_clean, ld_corr = logit_diff(clean_logits), logit_diff(corr_logits)
print("mean LD clean:", round(ld_clean.mean().item(), 2), " share LD > 0:", (ld_clean > 0).float().mean().item())
print("mean LD corrupted:", round(ld_corr.mean().item(), 2))

**Задание 1. Чему равна средняя разность логитов IO и S на 200 чистых промптах? Ответ округлите до сотых.**

## **Линейный зонд по слоям**

Зонд — логистическая регрессия на активациях residual stream одной позиции одного слоя. Слой $l$ здесь — вход блока $l$ (`blocks.l.hook_resid_pre`), слой 12 — выход последнего блока. Зонд учится на 120 промптах и проверяется на остальных 80; точность 0.5 — уровень случайного угадывания.

Допишите обучение зонда и подсчет точности на отложенных промптах.

In [ ]:
L = model.cfg.n_layers
perm = np.random.permutation(N)
train, test = perm[:120], perm[120:]


def resid(layer, pos):
    name = f"blocks.{layer}.hook_resid_pre" if layer < L else f"blocks.{L - 1}.hook_resid_post"
    return clean_cache[name][:, pos].numpy()


probe_acc = {}
for pos_name, pos in [("S2", S2), ("END", END)]:
    accs = []
    for layer in range(L + 1):
        h = resid(layer, pos)
        accs.append(...)  # Ваш код здесь
    probe_acc[pos_name] = accs
    print(pos_name, [round(a, 3) for a in accs])

**Задание 2. С какого слоя зонд на позиции END впервые различает метку `y` с точностью 1.0? Запишите номер слоя.**

**Задание 3. С какого слоя зонд на позиции S2 различает метку `y` с точностью 1.0? Запишите номер слоя.**

## **Activation patching по слоям**

Запускаем модель на испорченном промпте и на одной позиции одного слоя подставляем активацию residual stream с чистого промпта. Если эта активация несет то, что модель использует для ответа, разность логитов вернется к чистой. Доля восстановления

$$r_l = \frac{\overline{\mathrm{LD}}_{\text{patched}} - \overline{\mathrm{LD}}_{\text{corr}}}{\overline{\mathrm{LD}}_{\text{clean}} - \overline{\mathrm{LD}}_{\text{corr}}},$$

где черта — среднее по 200 промптам: $r_l = 1$ — подстановка восстановила ответ полностью, $r_l = 0$ — не изменила ничего.

Допишите хук, который подставляет активацию, и формулу $r_l$.

In [ ]:
patch_frac = {}
for pos_name, pos in [("S2", S2), ("END", END)]:
    fracs = []
    for layer in range(L):
        def patch(act, hook):
            ...  # Ваш код здесь
            return act

        patched = model.run_with_hooks(corr_tokens, fwd_hooks=[(f"blocks.{layer}.hook_resid_pre", patch)])
        ld_patched = logit_diff(patched)
        fracs.append(...)  # Ваш код здесь
    patch_frac[pos_name] = fracs
    print(pos_name, [round(f, 3) for f in fracs])

**Задание 4. С какого слоя patching на позиции S2 перестает восстанавливать ответ: $r_l$ впервые меньше 0.1? Запишите номер слоя.**

**Задание 5. Чему равна доля восстановления $r_l$ при patching на позиции END на том слое, который вы нашли в задании 2? Ответ округлите до сотых.**

## **Сравнение: декодируется против используется**

Сведем обе кривые в одну таблицу. Где зонд дает 1.0, а patching — около нуля, информация в активации есть, но модель берет ее для ответа не отсюда.

In [ ]:
print("layer  probe S2  patch S2  probe END  patch END")
for layer in range(L + 1):
    ps = f"{patch_frac['S2'][layer]:8.3f}" if layer < L else "       —"
    pe = f"{patch_frac['END'][layer]:9.3f}" if layer < L else "        —"
    print(f"{layer:5d}  {probe_acc['S2'][layer]:8.3f}  {ps}  {probe_acc['END'][layer]:9.3f}  {pe}")

**Задание 6. На позиции S2 на слоях 9–12 зонд различает метку с точностью 1.0, а patching восстанавливает почти ничего. Что из этого следует?**

## **Заключение**

- Зонд отвечает на вопрос, *есть ли* в активации информация: на позиции S2 метка `y` декодируется уже с первого слоя и остается декодируемой до выхода модели.
- Activation patching отвечает на вопрос, *пользуется ли* модель этой активацией: на позиции S2 подстановка восстанавливает ответ только до середины сети, а на позиции END — только с поздних слоев, когда туда уже перенесено имя для ответа.
- Слои, где зонд дает 1.0, а patching около нуля, — та разница между декодируемостью и причинной ролью, о которой говорит карта методов: высокая точность зонда сама по себе не доказывает, что модель пользуется представлением. Кейс Othello-GPT потому и проверял представление доски вмешательством.
- Что именно переносит информацию с позиции S2 на позицию END, показывает разбор схемы IOI по головам внимания (Wang et al., 2022): это темы раздела «Схемы (Circuits)».